# 01 Prepare — Data Readiness and Source Suitability

This notebook prepares the selected public dataset for the next project stage by:

- confirming the source and its safe-use limits;
- checking the required fields, raw grain, keys, basic quality, and time coverage; and
- recording one readiness decision and a clear Process handoff.

**Logic:** Ask requirements → raw-source evidence → readiness decision → Process handoff

The notebook profiles the raw data but does not clean, aggregate, model, or answer AQ1–AQ4.

## 1. Confirm the Prepare-stage scope

**Purpose:** Translate the decisions from Notebook 00 into the data requirements that must be checked before processing begins.

**Logic:** Ask decisions → required grains and fields → Prepare checks

**Confirm the Prepare-stage question**

**Can the selected public source support the analytical plan defined in `00_define_problems.ipynb`?**

The exit status is one of:

- `READY`;
- `CONDITIONALLY READY`; or
- `NOT READY`.

**Confirm the downstream analytical requirements**

| Requirement | Definition used in later notebooks |
|---|---|
| Decision grain | One eligible customer order |
| Raw source grain | One order item per row |
| Target | Binary order-level `Late_delivery_risk` |
| Prediction time | Order-creation timestamp |
| Predictor boundary | Only information available at order creation |
| Weekly structure | Complete Monday-to-Sunday weeks for weekly trends and the temporal model split |
| Review-capacity scenarios | Top-ranked 5%, 10%, and 20% of eligible held-out orders |
| Decision mode | Human decision support |

**Define the stage boundary**

- **Prepare:** review the source, identify limitations, and test readiness.
- **Process:** clean fields, define eligibility, aggregate to order level, and create the complete-week dataset.
- **Analyze:** answer AQ1–AQ4 using the two validated Process outputs.

## 2. Review the source and use boundaries

**Purpose:** Confirm where the data came from and define how it may be used safely in the portfolio.

**Logic:** Source identity → suitability limits → privacy boundary

### 2.1 Confirm source provenance and suitability

**Purpose:** Record the selected source and assess whether it is suitable for demonstrating the planned analytical workflow.

**Logic:** Source record → licence check → suitability decision

**Record the source details**

| Item | Details |
|---|---|
| Source | *DataCo SMART SUPPLY CHAIN FOR BIG DATA ANALYSIS* |
| Creators | Fabian Constante, Fernando Silva, and António Pereira |
| Repository | [Mendeley Data — Version 5](https://doi.org/10.17632/8gx2fvg2k6.5) |
| Version | 5 |
| DOI | [`10.17632/8gx2fvg2k6.5`](https://doi.org/10.17632/8gx2fvg2k6.5) |
| Licence | [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) |
| Access date | 4 August 2026 |
| Raw files | `DataCoSupplyChainDataset.csv`; `DescriptionDataCoSupplyChain.csv` |

**Assess source suitability**

- The source contains order, delivery, customer-segment, market, product, value, and date fields relevant to the planned analysis.
- It is suitable as public proxy data for demonstrating the workflow.
- It is not treated as representative evidence for another organisation or operating context.
- Results are descriptive and predictive within this dataset; they do not establish causality, deployment impact, savings, or return on investment.
- Version 5 is fixed for this project and is cited in the final handoff.

### 2.2 Define privacy and field-use boundaries

**Purpose:** Identify fields that must be removed or restricted before privacy-safe analytical outputs are created.

**Logic:** Sensitive fields → permitted key use → safe downstream outputs

**Classify sensitive fields and operational keys**

| Field group | Source fields | Prepare decision |
|---|---|---|
| Direct identifiers and credentials | `Customer Email`, `Customer Fname`, `Customer Lname`, `Customer Password`, `Customer Street` | Exclude from analytical and public outputs |
| Fine-grained locations | `Customer City`, `Customer Zipcode`, `Latitude`, `Longitude`, `Order City`, `Order Zipcode` | Exclude; retain only justified coarse geography |
| Operational keys | `Customer Id`, `Order Customer Id`, `Order Id`, `Order Item Id` | Use only for grain, joins, aggregation, and reconciliation |

- Raw source files remain outside the public portfolio repository.
- Only privacy-safe processed tables and analytical outputs are retained for later stages.

## 3. Load the source files

**Purpose:** Load the fixed source files from the project structure without changing or previewing sensitive raw rows.

**Logic:** Project paths → file check → controlled loading

**Define project paths and load the files**

In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display


# Identify the project root when the notebook runs from either
# the project root or the notebooks folder.
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


# Define the agreed raw-data paths.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
MAIN_PATH = RAW_DIR / "DataCoSupplyChainDataset.csv"
DICTIONARY_PATH = RAW_DIR / "DescriptionDataCoSupplyChain.csv"


# Confirm that both files are available before loading them.
required_paths = [MAIN_PATH, DICTIONARY_PATH]
missing_files = [path.name for path in required_paths if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Required raw file(s) not found in data/raw: "
        + ", ".join(missing_files)
    )


# Load the source files without modifying them.
raw_df = pd.read_csv(
    MAIN_PATH,
    encoding="latin-1",
    sep=",",
    low_memory=False,
)

dictionary_df = pd.read_csv(
    DICTIONARY_PATH,
    encoding="latin-1",
    sep=",",
)

if raw_df.empty or dictionary_df.empty:
    raise ValueError("One or more source files loaded without records.")


# Display only structural loading results.
print(
    f"Loaded main dataset: "
    f"{raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns"
)
print(
    f"Loaded field description: "
    f"{dictionary_df.shape[0]:,} rows × {dictionary_df.shape[1]} columns"
)
print("Raw row values are intentionally not previewed.")


def normalise_field_name(value: object) -> str:
    # Normalise labels for the schema comparison only.
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())

Loaded main dataset: 180,519 rows × 53 columns
Loaded field description: 52 rows × 2 columns
Raw row values are intentionally not previewed.


**Review the loading result**

- The main dataset loaded with **180,519 rows and 53 columns**.
- The field-description file loaded with **52 rows and 2 columns**.
- No raw row preview is required for the Prepare decision.

## 4. Confirm schema, required fields, grain, and keys

**Purpose:** Confirm that the raw structure can be transformed into the two datasets used by the current Analyze workflow.

**Logic:** Schema coverage → required fields → raw grain and keys → safe order-level aggregation

### 4.1 Compare the data dictionary with observed fields

**Purpose:** Identify source columns whose names are missing from either the raw dataset or its field description.

**Logic:** Normalise names → compare field sets → record gaps for Process

**Compare documented and observed fields**

In [2]:
required_dictionary_columns = {"FIELDS", "DESCRIPTION"}
if not required_dictionary_columns.issubset(dictionary_df.columns):
    raise ValueError(
        "The field-description file does not contain the expected FIELDS and DESCRIPTION columns."
    )

observed_fields = {normalise_field_name(column): column for column in raw_df.columns}
documented_fields = {
    normalise_field_name(field): str(field).strip()
    for field in dictionary_df["FIELDS"].dropna()
}

observed_not_documented = sorted(
    observed_fields[key] for key in observed_fields.keys() - documented_fields.keys()
)
documented_not_observed = sorted(
    documented_fields[key] for key in documented_fields.keys() - observed_fields.keys()
)

schema_comparison = pd.DataFrame([
    {"Measure": "Observed main-CSV fields", "Value": len(raw_df.columns)},
    {"Measure": "Documented fields", "Value": len(documented_fields)},
    {"Measure": "Observed but not documented", "Value": len(observed_not_documented)},
    {"Measure": "Documented but not observed", "Value": len(documented_not_observed)},
])

schema_gaps = pd.DataFrame([
    {
        "Gap type": "Observed but not documented",
        "Fields": ", ".join(observed_not_documented) or "None",
        "Process treatment": "Document locally and exclude if meaning or use is not justified",
    },
    {
        "Gap type": "Documented but not observed",
        "Fields": ", ".join(documented_not_observed) or "None",
        "Process treatment": "No action when none; otherwise resolve before use",
    },
])

display(schema_comparison)
display(schema_gaps)

,Measure,Value
0,Observed main-CSV fields,53
1,Documented fields,52
2,Observed but not documented,1
3,Documented but not observed,0


,Gap type,Fields,Process treatment
0,Observed but not documented,Order Zipcode,Document locally and exclude if meaning or use...
1,Documented but not observed,None,No action when none; otherwise resolve before use


**Review the schema findings**

- The main CSV contains **53 observed fields**.
- The field description contains **52 documented fields**.
- `Order Zipcode` is observed but not documented.
- No documented field is missing from the main CSV.
- `Order Zipcode` is already excluded under the fine-grained location rule, so the documentation gap does not block the project.

### 4.2 Confirm the required elements and Analyze dependencies

**Purpose:** Confirm that the source contains the elements defined in Ask and the fields required by the actual AQ1–AQ4 workflow.

**Logic:** E01–E10 requirements → source fields → AQ1–AQ4 inputs

**Map E01–E10 to the source**

| ID | Required element | Source evidence | Status |
|---|---|---|---|
| E01 | Order identifier | `Order Id` | Available |
| E02 | Order-item identifier | `Order Item Id` | Available |
| E03 | Order-creation timestamp | `order date (DateOrders)` | Available |
| E04 | Scheduled service | `Days for shipment (scheduled)`, `Shipping Mode` | Available |
| E05 | Observed delivery outcome | `Late_delivery_risk`, `Delivery Status`, `Days for shipping (real)` | Available |
| E06 | Shipping option | `Shipping Mode` | Available |
| E07 | Product mix, quantity, and value | `Product Card Id`, `Order Item Quantity`, `Sales`, `Order Item Total` | Available |
| E08 | Non-sensitive customer segment | `Customer Segment` | Available |
| E09 | Coarse market or region | `Market`, `Order Region`, `Order Country` | Available |
| E10 | Calendar fields | Derived from `order date (DateOrders)` | Available |

**Link the Prepare fields to AQ1–AQ4**

| Analyze question | Required Process input | Main fields or derived measures |
|---|---|---|
| AQ1 — Delivery performance | Order-level and complete-week datasets | Order date, late-delivery target, weekly counts, `Shipping Mode`, `Customer Segment`, `Market` |
| AQ2 — Associated factors | Order-level dataset | Segment fields, `Order Item Count`, `Order Quantity`, `Order Value` |
| AQ3 — Risk ranking | Order-level dataset plus complete-week boundaries | `Shipping Mode`, `Type`, `Customer Segment`, `Market`, `Order Item Count`, `Order Quantity`, `Order Value`, target, and order date |
| AQ4 — Prioritisation | AQ3 scored held-out orders | `Order Id`, risk score, actual target, and 5%/10%/20% capacity shares |

AQ4 does not require another raw-data structure; it reuses the AQ3 held-out risk ranking.

### 4.3 Confirm the raw grain and candidate keys

**Purpose:** Distinguish expected order-item repetition from true duplicate rows and identify the correct key at each grain.

**Logic:** Count raw rows and IDs → test candidate keys → define the order-level transformation

**Calculate the grain and key profile**

In [3]:
raw_rows = len(raw_df)
unique_orders = raw_df["Order Id"].nunique(dropna=True)
unique_order_items = raw_df["Order Item Id"].nunique(dropna=True)

items_per_order = raw_df.groupby("Order Id", dropna=False).size()

grain_profile = pd.DataFrame([
    {"Measure": "Raw rows", "Value": raw_rows, "Interpretation": "Order-item records"},
    {"Measure": "Unique Order Id", "Value": unique_orders, "Interpretation": "Target decision grain"},
    {"Measure": "Unique Order Item Id", "Value": unique_order_items, "Interpretation": "Candidate raw-row key"},
    {"Measure": "Orders with more than one item row", "Value": int((items_per_order > 1).sum()), "Interpretation": "Expected repeated line-item structure"},
    {"Measure": "Maximum item rows per order", "Value": int(items_per_order.max()), "Interpretation": "Observed line-item depth"},
])

candidate_keys = pd.DataFrame([
    {
        "Candidate key": "Order Item Id",
        "Intended grain": "Raw order item",
        "Missing values": int(raw_df["Order Item Id"].isna().sum()),
        "Duplicate rows": int(raw_df.duplicated("Order Item Id").sum()),
        "Decision": "VALID" if raw_df["Order Item Id"].notna().all() and raw_df["Order Item Id"].is_unique else "REVIEW",
    },
    {
        "Candidate key": "Order Id",
        "Intended grain": "Unique order after aggregation",
        "Missing values": int(raw_df["Order Id"].isna().sum()),
        "Duplicate rows": int(raw_df.duplicated("Order Id").sum()),
        "Decision": "EXPECTED REPEATS IN RAW DATA",
    },
    {
        "Candidate key": "Order Id + Order Item Cardprod Id",
        "Intended grain": "Order-product combination",
        "Missing values": int(raw_df[["Order Id", "Order Item Cardprod Id"]].isna().any(axis=1).sum()),
        "Duplicate rows": int(raw_df.duplicated(["Order Id", "Order Item Cardprod Id"]).sum()),
        "Decision": "NOT A RAW-ROW KEY",
    },
])

display(grain_profile)
display(candidate_keys)

,Measure,Value,Interpretation
0,Raw rows,180519,Order-item records
1,Unique Order Id,65752,Target decision grain
2,Unique Order Item Id,180519,Candidate raw-row key
3,Orders with more than one item row,45902,Expected repeated line-item structure
4,Maximum item rows per order,5,Observed line-item depth


,Candidate key,Intended grain,Missing values,Duplicate rows,Decision
0,Order Item Id,Raw order item,0,0,VALID
1,Order Id,Unique order after aggregation,0,114767,EXPECTED REPEATS IN RAW DATA
2,Order Id + Order Item Cardprod Id,Order-product combination,0,20756,NOT A RAW-ROW KEY


**Review the grain and key findings**

- The **180,519 raw rows** represent order-item records.
- They contain **65,752 unique orders** and **180,519 unique order items**.
- `Order Item Id` is complete and unique, so it is the valid raw-row key.
- Repeated `Order Id` values are expected because one order can contain up to five item rows.
- Process must aggregate item rows to one eligible order before Analyze begins.

### 4.4 Confirm order-level outcome consistency

**Purpose:** Confirm that fields retained once per order do not conflict across the order's item rows.

**Logic:** Group item rows by `Order Id` → count conflicting values → assess aggregation safety

**Check order-level fields across item rows**

In [4]:
order_level_fields = [
    "Late_delivery_risk",
    "Delivery Status",
    "Days for shipment (scheduled)",
    "Days for shipping (real)",
    "order date (DateOrders)",
    "Shipping Mode",
]

consistency_rows = []
for field in order_level_fields:
    distinct_per_order = raw_df.groupby("Order Id", dropna=False)[field].nunique(dropna=False)
    inconsistent_orders = int((distinct_per_order > 1).sum())
    consistency_rows.append({
        "Field": field,
        "Inconsistent orders": inconsistent_orders,
        "Status": "PASS" if inconsistent_orders == 0 else "REVIEW",
    })

outcome_consistency = pd.DataFrame(consistency_rows)

order_outcomes = raw_df[["Order Id", "Late_delivery_risk"]].drop_duplicates("Order Id")
late_counts = order_outcomes["Late_delivery_risk"].value_counts(dropna=False).sort_index()
target_profile = pd.DataFrame([
    {
        "Order-level target": str(label),
        "Orders": int(count),
        "Share": count / len(order_outcomes),
    }
    for label, count in late_counts.items()
])

display(outcome_consistency)
display(target_profile)

,Field,Inconsistent orders,Status
0,Late_delivery_risk,0,PASS
1,Delivery Status,0,PASS
2,Days for shipment (scheduled),0,PASS
3,Days for shipping (real),0,PASS
4,order date (DateOrders),0,PASS
5,Shipping Mode,0,PASS


,Order-level target,Orders,Share
0,0,29704,0.451758
1,1,36048,0.548242


**Review the consistency findings**

- No conflicting order-level values were found for the target, delivery status, scheduled days, actual days, order date, or shipping mode.
- The raw source contains **36,048 late orders** and **29,704 non-late orders** before Process eligibility rules are applied.
- These are raw-source counts; the final Analyze denominator is defined after Process filters and aggregation.

## 5. Profile the basic raw-data quality

**Purpose:** Identify the limited set of data-quality issues that Process must address before creating analysis-ready outputs.

**Logic:** Missingness and duplicates → date and value checks → Process treatment requirements

**Calculate the basic quality profile**

In [5]:
missing_profile = (
    raw_df.isna()
    .sum()
    .rename("Missing rows")
    .to_frame()
    .assign(
        **{"Missing share": lambda frame: frame["Missing rows"] / len(raw_df)}
    )
    .query("`Missing rows` > 0")
    .sort_values("Missing rows", ascending=False)
    .reset_index(names="Field")
)

duplicate_profile = pd.DataFrame([
    {"Check": "Exact duplicate raw rows", "Affected rows": int(raw_df.duplicated().sum()), "Interpretation": "Unexpected if greater than zero"},
    {"Check": "Duplicate Order Item Id", "Affected rows": int(raw_df.duplicated("Order Item Id").sum()), "Interpretation": "Would invalidate the candidate raw-row key"},
    {"Check": "Repeated Order Id", "Affected rows": int(raw_df.duplicated("Order Id").sum()), "Interpretation": "Expected because one order can contain several item rows"},
])

order_timestamp = pd.to_datetime(raw_df["order date (DateOrders)"], errors="coerce")
shipping_timestamp = pd.to_datetime(raw_df["shipping date (DateOrders)"], errors="coerce")

parsing_profile = pd.DataFrame([
    {"Field": "order date (DateOrders)", "Parsing failures": int(order_timestamp.isna().sum()), "Minimum": order_timestamp.min(), "Maximum": order_timestamp.max()},
    {"Field": "shipping date (DateOrders)", "Parsing failures": int(shipping_timestamp.isna().sum()), "Minimum": shipping_timestamp.min(), "Maximum": shipping_timestamp.max()},
])

invalid_value_profile = pd.DataFrame([
    {"Rule": "Order Item Quantity must be greater than zero", "Affected rows": int((raw_df["Order Item Quantity"] <= 0).sum())},
    {"Rule": "Sales must be non-negative", "Affected rows": int((raw_df["Sales"] < 0).sum())},
    {"Rule": "Order Item Total must be non-negative", "Affected rows": int((raw_df["Order Item Total"] < 0).sum())},
    {"Rule": "Order Item Discount Rate must be between 0 and 1", "Affected rows": int((~raw_df["Order Item Discount Rate"].between(0, 1)).sum())},
    {"Rule": "Scheduled shipping days must be non-negative", "Affected rows": int((raw_df["Days for shipment (scheduled)"] < 0).sum())},
    {"Rule": "Actual shipping days must be non-negative", "Affected rows": int((raw_df["Days for shipping (real)"] < 0).sum())},
    {"Rule": "Late_delivery_risk must be binary", "Affected rows": int((~raw_df["Late_delivery_risk"].isin([0, 1])).sum())},
    {"Rule": "Shipping timestamp must not precede order timestamp", "Affected rows": int((shipping_timestamp < order_timestamp).sum())},
])

display(missing_profile if not missing_profile.empty else pd.DataFrame([{"Finding": "No missing values"}]))
display(duplicate_profile)
display(parsing_profile)
display(invalid_value_profile)

,Field,Missing rows,Missing share
0,Product Description,180519,1.000000
1,Order Zipcode,155679,0.862397
2,Customer Lname,8,0.000044
3,Customer Zipcode,3,0.000017


,Check,Affected rows,Interpretation
0,Exact duplicate raw rows,0,Unexpected if greater than zero
1,Duplicate Order Item Id,0,Would invalidate the candidate raw-row key
2,Repeated Order Id,114767,Expected because one order can contain several...


,Field,Parsing failures,Minimum,Maximum
0,order date (DateOrders),0,2015-01-01,2018-01-31 23:38:00
1,shipping date (DateOrders),0,2015-01-03,2018-02-06 22:14:00


,Rule,Affected rows
0,Order Item Quantity must be greater than zero,0
1,Sales must be non-negative,0
2,Order Item Total must be non-negative,0
3,Order Item Discount Rate must be between 0 and 1,0
4,Scheduled shipping days must be non-negative,0
5,Actual shipping days must be non-negative,0
6,Late_delivery_risk must be binary,0
7,Shipping timestamp must not precede order time...,0


**Review the quality findings**

- Missing values occur in four fields: `Product Description`, `Order Zipcode`, `Customer Lname`, and `Customer Zipcode`.
- The source contains **0 exact duplicate rows** and **0 duplicate `Order Item Id` values**.
- Repeated `Order Id` values reflect the expected item-level grain.
- Both date fields parse without failures, and no shipping timestamp precedes its order timestamp.
- The selected quantity, value, discount, shipping-day, and binary-target rules return no invalid rows.
- Missing-value treatment and field removal remain Process tasks.

## 6. Check field timing and complete-week coverage

**Purpose:** Protect AQ3 from post-order information and confirm the complete-week structure used by AQ1 and the temporal model split.

**Logic:** Field timing → leakage-safe use → complete-week coverage

### 6.1 Define field use at order creation

**Purpose:** Classify only the fields needed by the current Analyze design instead of maintaining a broad 53-field predictor register.

**Logic:** Decision time → permitted role → AQ3 feature set

**Classify the fields used by Analyze**

| Field role | Fields | Use rule |
|---|---|---|
| Trace key | `Order Id` | Retain for order tracing; do not use as a predictor |
| Time field | `order date (DateOrders)` | Define complete weeks and the chronological split |
| AQ3 predictors | `Shipping Mode`, `Type`, `Customer Segment`, `Market`, `Order Item Count`, `Order Quantity`, `Order Value` | Use because they are available at order creation or are derived from order-creation item records |
| Target | `Late_delivery_risk` | Use as the outcome only |
| Retrospective delivery fields | `Days for shipping (real)`, `Delivery Status`, `shipping date (DateOrders)` | Use for historical outcomes or validation; exclude from AQ3 predictors |
| Sensitive fields | Direct identifiers, credentials, fine-grained locations | Exclude from analytical outputs |
| Other unclear or unused fields | Profit, later status, image, and description fields not required by AQ1–AQ4 | Exclude from the current analysis |

This compact classification matches the fields actually selected in `03_analyze_delivery_risk_and_workload.ipynb`.

### 6.2 Confirm date coverage and complete weeks

**Purpose:** Confirm that unique orders form a continuous sequence of complete Monday-to-Sunday weeks.

**Logic:** Parse order dates → derive week starts → separate boundary weeks → check continuity

**Calculate the complete-week coverage**

In [6]:
order_dates = (
    raw_df[["Order Id", "order date (DateOrders)"]]
    .drop_duplicates("Order Id")
    .assign(order_timestamp=lambda frame: pd.to_datetime(frame["order date (DateOrders)"], errors="coerce"))
)

if order_dates["order_timestamp"].isna().any():
    raise ValueError("Order-creation timestamps contain parsing failures; temporal readiness cannot be assessed.")

order_dates["week_start"] = (
    order_dates["order_timestamp"].dt.normalize()
    - pd.to_timedelta(order_dates["order_timestamp"].dt.weekday, unit="D")
)

weekly_order_counts = order_dates.groupby("week_start").size().sort_index()
minimum_date = order_dates["order_timestamp"].min()
maximum_date = order_dates["order_timestamp"].max()

complete_week_counts = weekly_order_counts.loc[
    (weekly_order_counts.index >= minimum_date.normalize())
    & (weekly_order_counts.index + pd.Timedelta(days=6) <= maximum_date.normalize())
]

expected_complete_weeks = pd.date_range(
    complete_week_counts.index.min(),
    complete_week_counts.index.max(),
    freq="W-MON",
)
missing_complete_weeks = expected_complete_weeks.difference(complete_week_counts.index)
boundary_weeks = weekly_order_counts.index.difference(complete_week_counts.index)

temporal_readiness = pd.DataFrame([
    {"Measure": "Earliest order timestamp", "Value": minimum_date},
    {"Measure": "Latest order timestamp", "Value": maximum_date},
    {"Measure": "Observed weekly periods including boundaries", "Value": len(weekly_order_counts)},
    {"Measure": "Complete Monday-to-Sunday weeks", "Value": len(complete_week_counts)},
    {"Measure": "Missing weeks inside the complete range", "Value": len(missing_complete_weeks)},
    {"Measure": "Partial boundary weeks", "Value": len(boundary_weeks)},
    {"Measure": "Complete weekly sequence available", "Value": bool(len(complete_week_counts) > 0 and len(missing_complete_weeks) == 0)},
])

display(temporal_readiness)

,Measure,Value
0,Earliest order timestamp,2015-01-01 00:00:00
1,Latest order timestamp,2018-01-31 23:38:00
2,Observed weekly periods including boundaries,162
3,Complete Monday-to-Sunday weeks,160
4,Missing weeks inside the complete range,0
5,Partial boundary weeks,2
6,Complete weekly sequence available,True


**Review the temporal findings**

- Order timestamps cover **1 January 2015 to 31 January 2018**.
- The source contains **160 complete Monday-to-Sunday weeks**.
- There are **0 missing weeks** inside the complete range and **2 partial boundary weeks**.
- The complete-week sequence supports AQ1 weekly trends and the chronological AQ3 training/test split.

## 7. Decide readiness and hand off to Process

**Purpose:** Convert the Prepare evidence into one readiness decision and the minimum Process actions required by Analyze.

**Logic:** Evidence review → readiness status → Process actions → Analyze inputs

### 7.1 Record the readiness decision

**Purpose:** Separate true blockers from manageable conditions without generating an unnecessary decision matrix in code.

**Logic:** Required checks → blocking assessment → final status

**Apply the readiness decision**

| Readiness check | Result |
|---|---|
| Both Version 5 files load successfully | Pass |
| E01–E10 and current Analyze fields are available | Pass |
| `Order Item Id` is a complete unique raw-row key | Pass |
| Repeated order-level outcome and service fields are consistent | Pass |
| Order dates parse and form a continuous complete-week sequence | Pass |
| Sensitive fields and missing values require Process treatment | Condition |
| The public proxy is not representative of another operating context | Condition |

**Prepare status: `CONDITIONALLY READY`**

**Record the active conditions**

- Remove direct identifiers and fine-grained location fields from analytical outputs.
- Treat the documented missing values and the `Order Zipcode` dictionary gap.
- Apply the agreed eligibility rule and aggregate item rows to one order.
- Restrict AQ3 predictors to information available at order creation.
- Carry the public-proxy limitation into later interpretation.

### 7.2 Define the Process and Analyze handoff

**Purpose:** Translate the Prepare findings into the minimum transformations and saved data structures required by the current Analyze notebook.

**Logic:** Prepare conditions → Process transformations → two validated outputs → AQ1–AQ4

**Define the Process actions**

1. Read the raw files without overwriting them and save processed outputs separately.
2. Remove sensitive, credential, and fine-grained location fields.
3. Standardise required names and types and apply the documented quality treatments.
4. Apply the eligibility rule and aggregate item rows to one order, creating `Order Item Count`, `Product Count`, `Order Quantity`, and `Order Value`.
5. Retain the required target, trace, segment, service, and date fields while excluding post-order information from AQ3 predictors.
6. Build a weekly table from unique eligible orders and retain complete Monday-to-Sunday weeks.

**Confirm the saved inputs used by Analyze**

| Process output | Grain | Use in `03 Analyze` |
|---|---|---|
| `data/processed/order_level_analysis_ready.csv` | One eligible order per row | AQ1 overall, monthly, and segment results; AQ2 comparisons; AQ3 features, target, and temporal split; AQ4 order tracing |
| `data/processed/weekly_analysis_ready.csv` | One complete Monday-to-Sunday week per row | AQ1 weekly trends and AQ3 complete-week split boundaries |

**Complete the Prepare handoff**

The selected source is suitable to proceed to `02_process_data_cleaning_and_validation.ipynb` within the stated conditions. The decision does not mean the raw data is already analysis-ready; Process must still create and validate the two saved inputs used by AQ1–AQ4.

**Source reference:** Constante, F., Silva, F., & Pereira, A. (2019). *DataCo SMART SUPPLY CHAIN FOR BIG DATA ANALYSIS* (Version 5). Mendeley Data. https://doi.org/10.17632/8gx2fvg2k6.5